# تقييم التلخيص التوليدي على Arabic XL-Sum

يوثق الدفتر تقييم نموذج 1200 خطوة ومقارنته بنموذج 200 خطوة دون تدريب أو تعديل أوزان. يتطلب الوصول المصرح إلى ملفات النموذجين، مع استبدال أمثلة المسارات العامة بمواقع التخزين المحلية.

تتضمن عينة test مقدار 300 مقال ببذرة 42، ومنها عينة 64 التي فُحصت سابقًا؛ لذلك تُعد المقارنة تقييمًا وصفيًا موسعًا وليست اختبارًا مستقلًا تمامًا عن قرارات التطوير. تتضمن validation مقدار 100 مقال منفصل. تُستخدم أمثلة وإعدادات توليد متطابقة للنموذجين، مع ROUGE وفروق paired bootstrap ومؤشرات وصفية. لا تُقاس صحة الوقائع آليًا بهذه المقاييس.


## 1. الإعدادات والمكتبات


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['ACCELERATE_MIXED_PRECISION'] = 'no'

TEST_SAMPLES = 300
VALIDATION_SAMPLES = 100
SEED = 42
SOURCE_LENGTH, TARGET_LENGTH = 1024, 128
NUM_BEAMS = 4
COMPARE_200 = True
EVALUATE_VALIDATION = True
BOOTSTRAP_ROUNDS = 1000

# مثال عام؛ يتطلب تحديد مجلد النموذج المصرح به قبل التنفيذ
MODEL_1200 = Path('/content/drive/MyDrive/AUTHORIZED_PROJECT')
DRIVE_ROOT = Path('/content/drive/MyDrive/AUTHORIZED_PROJECT')
WORK_DIR = Path('/content/arabic-evaluation')
WORK_DIR.mkdir(parents=True, exist_ok=True)

# حافظي على PyTorch/CUDA الموجود في Colab؛ تجنّبي خطأ TensorFlow/Protobuf السابق.
PINS = ['transformers==4.57.1','accelerate==1.11.0','datasets==4.3.0',
        'huggingface-hub==0.36.0','sentencepiece==0.2.1','protobuf==6.31.1',
        'safetensors==0.6.2','rouge-score==0.1.2','trafilatura==2.0.0',
        'urllib3==2.5.0']
subprocess.run([sys.executable,'-m','pip','install','-q',*PINS],check=True)
print('Setup complete; no training components installed or run.')

## 2. الوصول إلى النماذج ومصدر الشيفرة المثبّت


In [ ]:
import json, gc, time, csv, statistics, re
import numpy as np
import torch
from google.colab import drive
from huggingface_hub import HfApi
from types import SimpleNamespace
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm.auto import tqdm

drive.mount('/content/drive')
if not torch.cuda.is_available():
    raise RuntimeError('اختاري Runtime → Change runtime type → T4 GPU')
print('GPU:',torch.cuda.get_device_name(0))
if not (MODEL_1200 / 'model.safetensors').exists():
    raise FileNotFoundError(f'لم أجد أوزان نموذج 1200 خطوة: {MODEL_1200}')

def export_info(folder):
    p = folder / 'inference_config.json'
    if not p.exists(): return {}
    try:return json.loads(p.read_text(encoding='utf-8'))
    except Exception:return {}

info1200=export_info(MODEL_1200)
print('1200-step export metadata:', {k:info1200.get(k) for k in ('completed_steps','dataset_revision','model_revision','seed')})

# اكتشاف آمن: لا نختار نموذجاً اعتماداً على الاسم فقط؛ نتحقق من completed_steps.
candidates=[]
for metadata in DRIVE_ROOT.glob('*/model/inference_config.json'):
    if metadata.parent == MODEL_1200:continue
    data=export_info(metadata.parent)
    if data.get('completed_steps')==200 and (metadata.parent/'model.safetensors').is_file():
        candidates.append(metadata.parent)
MODEL_200 = candidates[0] if len(candidates)==1 else None
if COMPARE_200 and MODEL_200 is None:
    print('تعذر اختيار نموذج 200 خطوة تلقائياً (عدد المرشحين:',len(candidates),').')
    print('المرشحون:',list(map(str,candidates)))
    print('اختياري: عيّني MODEL_200 يدوياً إلى مجلد النموذج الصحيح إذا أردتِ المقارنة.')
else:
    print('Model 200:',MODEL_200)

REPO_DIR=WORK_DIR/'source'
SOURCE_COMMIT='d1b24cbeef87aa61e45693fd5f7895a9f8db6f05'
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','-q','--no-checkout','https://github.com/nada-ali-13/Arabic-news-summarizer.git',str(REPO_DIR)],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
sys.path.insert(0,str(REPO_DIR/'src'))
from arabic_summarizer.data import load_official, fixed_subset
from arabic_summarizer.metrics import rouge
from arabic_summarizer.text import normalize
print('Loaded the original Arabic preprocessing and ROUGE scoring implementation.')

## 3. عينات التقييم وإعدادات التوليد الموحدة


In [ ]:
DATASET_ID='csebuetnlp/xlsum'
revision = info1200.get('dataset_revision') or HfApi().dataset_info(DATASET_ID).sha
config=SimpleNamespace(dataset_id=DATASET_ID,dataset_revision=revision,
                       source_length=SOURCE_LENGTH,target_length=TARGET_LENGTH)
dataset=load_official(config,WORK_DIR/'arabic-data')
test_rows=list(fixed_subset(dataset['test'],TEST_SAMPLES,SEED))
validation_rows=list(fixed_subset(dataset['validation'],VALIDATION_SAMPLES,SEED)) if EVALUATE_VALIDATION else []
assert len(test_rows)==TEST_SAMPLES
if len({r['id'] for r in test_rows}) != len(test_rows):
    print('Notice: official test split retains duplicate IDs by design; rows are compared by deterministic position.')
print('Test:',len(test_rows),'Validation:',len(validation_rows))
print('Seed:',SEED,'same seeded test selection as the training notebooks, extended to 300.')

## 4. التوليد والتقييم

يُقيّم نموذج 1200 خطوة أولًا، ثم نموذج 200 خطوة اختياريًا على الأمثلة نفسها وبالإعدادات ذاتها. إذا تعذر الوصول إلى النموذج الأقدم، تبقى نتائج النموذج النهائي متاحة دون مقارنة زوجية.


In [ ]:
def make_predictions(model_dir, rows, label):
    if not model_dir or not model_dir.is_dir():raise FileNotFoundError(model_dir)
    tok=AutoTokenizer.from_pretrained(str(model_dir),use_fast=False,local_files_only=True)
    mod=AutoModelForSeq2SeqLM.from_pretrained(str(model_dir),dtype=torch.float32,
                    use_safetensors=True,weights_only=True,local_files_only=True).to('cuda').eval()
    mod.config.use_cache=True
    out=[]
    tic=time.monotonic()
    try:
        for r in tqdm(rows,desc=label):
            inp=tok(normalize(r['text']),max_length=SOURCE_LENGTH,truncation=True,return_tensors='pt')
            inp={k:v.to('cuda') for k,v in inp.items()}
            with torch.inference_mode():
                ids=mod.generate(**inp,max_new_tokens=TARGET_LENGTH,min_new_tokens=8,
                                 num_beams=NUM_BEAMS,do_sample=False,no_repeat_ngram_size=3,
                                 repetition_penalty=1.1,length_penalty=1.0,early_stopping=True)
            pred=normalize(tok.decode(ids[0],skip_special_tokens=True))
            out.append({'id':r['id'],'source':r['text'],'reference':r['summary'],'prediction':pred})
    finally:
        del mod,tok
        gc.collect();torch.cuda.empty_cache()
    print(label,'seconds:',round(time.monotonic()-tic,1))
    return out

results={}
results['1200_test']=make_predictions(MODEL_1200,test_rows,'1200-step test')
if EVALUATE_VALIDATION:
    results['1200_validation']=make_predictions(MODEL_1200,validation_rows,'1200-step validation')
if COMPARE_200 and MODEL_200 is not None:
    assert MODEL_200.resolve()!=MODEL_1200.resolve()
    results['200_test']=make_predictions(MODEL_200,test_rows,'200-step test')
    if EVALUATE_VALIDATION:
        results['200_validation']=make_predictions(MODEL_200,validation_rows,'200-step validation')
print('Models/datasets evaluated:',list(results))

## 5. ROUGE وفواصل الثقة

الفرق بالنقاط هو طرح درجتي ROUGE، والتحسن النسبي هو نسبة الفرق إلى الدرجة السابقة. تُقدّر فواصل الثقة بواسطة paired bootstrap على المقالات نفسها؛ لا تقيس صحة الوقائع أو الهلوسة.


In [ ]:
metrics={}
for name,records in results.items():
    metrics[name]=rouge([r['prediction'] for r in records],[r['reference'] for r in records])
    print(name,'count=',len(records),{k:round(v,3) for k,v in metrics[name].items()})

def paired_ci(records_new,records_old,key,rounds=1000):
    from rouge_score.rouge_scorer import RougeScorer
    from arabic_summarizer.metrics import ArabicTokenizer
    from arabic_summarizer.text import sentences
    assert [r['id'] for r in records_new]==[r['id'] for r in records_old]
    scorer=RougeScorer([key],tokenizer=ArabicTokenizer(),use_stemmer=False,split_summaries=False)
    def score(pred,ref):
        return scorer.score('\n'.join(sentences(normalize(ref))),
                            '\n'.join(sentences(normalize(pred))))[key].fmeasure*100
    delta=np.array([score(a['prediction'],a['reference'])-score(b['prediction'],b['reference'])
                  for a,b in zip(records_new,records_old)])
    rng=np.random.default_rng(SEED)
    indices=rng.integers(0,len(delta),size=(rounds,len(delta)))
    return {'point_delta':float(delta.mean()),'ci95_points':np.percentile(delta[indices].mean(axis=1),[2.5,97.5]).round(3).tolist()}

comparisons={}
for split in ('test','validation'):
    a,b=f'1200_{split}',f'200_{split}'
    if a in results and b in results:
        comparisons[split]={}
        for k in ('rouge1','rouge2','rougeL','rougeLsum'):
            old,new=metrics[b][k],metrics[a][k]
            comparisons[split][k]={
                'old':old,'new':new,'delta_points':new-old,
                'relative_improvement_percent':(100*(new-old)/old if old else None),
                **paired_ci(results[a],results[b],k,BOOTSTRAP_ROUNDS)
            }
        print('\nPAIRWISE COMPARISON',split)
        for k,v in comparisons[split].items():
            print(k,'old',round(v['old'],3),'new',round(v['new'],3),
                  'delta points',round(v['delta_points'],3),
                  'relative %',round(v['relative_improvement_percent'],2) if v['relative_improvement_percent'] is not None else 'n/a',
                  'paired bootstrap 95% CI',v['ci95_points'])
if not comparisons:
    print('No 200-step weights available: do not compare 300-row scores to the old 64-row result as a paired experiment.')

## 6. مؤشرات وصفية وعينات للمراجعة البشرية

مؤشرات التكرار والطول لا تعادل دقة التلخيص. تشمل المراجعة البشرية هوية الجهات والأسماء والأعداد والتواريخ والعلاقات السببية. لا تتضمن الحزمة تقييمًا بشريًا مكتملًا للوقائع.


In [ ]:
def repetition_rate(text):
    words=normalize(text).split()
    pairs=list(zip(words,words[1:]))
    return 0.0 if not pairs else (len(pairs)-len(set(pairs)))/len(pairs)

def diagnostics(records):
    return {
        'avg_source_words':float(np.mean([len(r['source'].split()) for r in records])),
        'avg_reference_words':float(np.mean([len(r['reference'].split()) for r in records])),
        'avg_prediction_words':float(np.mean([len(r['prediction'].split()) for r in records])),
        'empty_predictions':sum(not r['prediction'].strip() for r in records),
        'mean_repeated_bigram_fraction':float(np.mean([repetition_rate(r['prediction']) for r in records])),
    }
for name,records in results.items():
    print(name,diagnostics(records))

from IPython.display import HTML,display
from html import escape
review=results['1200_test']
# تنويع العينات بذر ثابت؛ لا يُنتقى فقط أفضل ما يفعله النموذج.
rng=np.random.default_rng(SEED)
review_indices=sorted(rng.choice(len(review),size=min(12,len(review)),replace=False).tolist())
for idx in review_indices[:5]:
    r=review[idx]
    display(HTML('<section dir="rtl" style="line-height:1.7;border-bottom:1px solid #ddd;padding:1em">'
                 f'<b>العينة {idx} — {escape(str(r["id"]))}</b>'
                 '<h4>المقال</h4><p>'+escape(r['source'][:3500])+'</p>'
                 '<h4>الملخص المرجعي</h4><p>'+escape(r['reference'])+'</p>'
                 '<h4>ملخص النموذج</h4><p>'+escape(r['prediction'])+'</p></section>'))

audit_rows=[{'id':review[i]['id'],'source':review[i]['source'],
             'reference':review[i]['reference'],'prediction':review[i]['prediction'],
             'factual_errors':'','omitted_key_facts':'','repetition':'','language_quality':'',
             'human_notes':''} for i in review_indices]
print('Prepared',len(audit_rows),'samples for human fact-checking. Blank fields require manual review.')

## 7. حفظ تقرير التقييم

يُحفظ التقرير في مجلد مستقل دون تعديل أوزان النموذجين. يُنشأ CSV للمراجعة البشرية؛ لا تعني الحقول الفارغة اكتمال تلك المراجعة.


In [ ]:
from datetime import datetime,timezone
report_dir=DRIVE_ROOT/'evaluation-reports'/('eval-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
report_dir.mkdir(parents=True,exist_ok=False)
summary={
  'protocol':{'dataset':'Arabic XL-Sum v2','test_n':len(test_rows),
              'validation_n':len(validation_rows),'seed':SEED,
              'source_length':SOURCE_LENGTH,'target_length':TARGET_LENGTH,
              'num_beams':NUM_BEAMS,'generative ROUGE not accuracy':True,
              'test_reused_for_prior_64_result':True},
  'models':{'1200':str(MODEL_1200),'200':str(MODEL_200) if '200_test' in results else None},
  'rouge':metrics,'comparisons':comparisons,
  'diagnostics':{k:diagnostics(v) for k,v in results.items()},
  'warning':'ROUGE cannot establish factual correctness. Inspect review CSV manually; test already inspected previously.'
}
(report_dir/'metrics_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
for label,rows in results.items():
    (report_dir/(label+'_predictions.json')).write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding='utf-8')
with (report_dir/'human_fact_check.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(audit_rows[0].keys()))
    w.writeheader();w.writerows(audit_rows)
print('EVALUATION COMPLETE')
print('Results in Google Drive:',report_dir)
print('No models were trained, overwritten or deleted.')